# AIFlow — reviewed parser-free FASHN virtual try-on
Use a fresh GPU Colab runtime, Python 3.12. Stop any SDXL worker before loading VTON: only one model process may use the GPU. Ampere/L4 supports the official BF16 path; T4 uses the official FP32 fallback and still needs a measured memory/quality check.

Upload only the matching reviewed `aiflow-vton-worker.zip`. The worker uses hash-locked wheels and immutable local safetensors/ONNX models, with offline inference. No human parser, noncommercial SegFormer weights, custom nodes, pickle checkpoints, git installation or remote Python is included.

Inputs: fictional adult person, shop garment product shot on a plain background. Only flat-lay garments are supported. Output is the full native 576x864 canvas; text prompts are recorded but do not condition FASHN. Inspect garment details and face/body preservation before publishing. Local tests are not GPU execution proof.

Every API route requires a private session Bearer token. The optional reviewed tunnel exposes transport only. Keep tokens out of notebook output and Drive. Jobs default to temporary `/content`; choose a trusted durable non-code storage folder only if you need reconnect persistence. This notebook never mounts Drive automatically.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import zipfile, hashlib

uploaded = files.upload()
if set(uploaded) != {'aiflow-vton-worker.zip'}:
    raise RuntimeError('Upload exactly the reviewed aiflow-vton-worker.zip')
if hashlib.sha256(Path('aiflow-vton-worker.zip').read_bytes()).hexdigest() != 'dc576fe1586d72e9c0fe1878ebf0b19f598c1ff4bf95300b6c513285fb658e3a':
    raise RuntimeError('Bundle differs from reviewed notebook SHA256')
CODE_ROOT = Path('/content/aiflow-vton-code')
CODE_ROOT.mkdir(exist_ok=True)
allowed = {'colab/vton_worker/__main__.py', 'colab/vton_worker/vendor/fashn_vton/utils/__init__.py', 'colab/vton_worker/models.py', 'colab/image_worker/__init__.py', 'colab/vton_worker/vendor/fashn_vton/utils/checkpoint.py', 'colab/vton_worker/vendor/fashn_vton/utils/common.py', 'colab/vton_worker/schema.py', 'colab/vton_worker/vendor/fashn_vton/LICENSE', 'colab/vton_worker/vendor/fashn_vton/utils/keypoints.py', 'colab/vton_worker/vendor/fashn_vton/preprocessing/transforms.py', 'colab/vton_worker/pipeline.py', 'colab/vton_worker/vendor/fashn_vton/utils/sampling.py', 'colab/vton_worker/requirements-lock.txt', 'colab/vton_worker/vendor/fashn_vton/dwpose/onnxdet.py', 'colab/vton_worker/vendor/fashn_vton/dwpose/dwpose.py', 'colab/vton_worker/source-lock.json', 'colab/image_worker/app.py', 'colab/vton_worker/__init__.py', 'colab/vton_worker/vendor/fashn_vton/dwpose/__init__.py', 'colab/vton_worker/vendor/fashn_vton/__init__.py', 'colab/vton_worker/vendor/fashn_vton/tryon_mmdit.py', 'colab/vton_worker/vendor/fashn_vton/NOTICE', 'colab/vton_worker/vendor/fashn_vton/utils/logger.py', 'colab/vton_worker/vendor/fashn_vton/preprocessing/__init__.py', 'colab/vton_worker/vendor/fashn_vton/dwpose/utils.py', 'colab/vton_worker/vendor/fashn_vton/utils/tensor.py', 'colab/vton_worker/engine.py', 'colab/vton_worker/audit-approval.json', 'colab/image_worker/models.py', 'colab/vton_worker/vendor/__init__.py', 'colab/vton_worker/model-lock.json', 'colab/vton_worker/bundle.py', 'colab/image_worker/bootstrap.py', 'colab/vton_worker/vendor/fashn_vton/dwpose/onnxpose.py', 'colab/vton_worker/README.md', 'colab/vton_worker/vendor/fashn_vton/dwpose/wholebody.py', 'colab/vton_worker/bootstrap.py'}
with zipfile.ZipFile('aiflow-vton-worker.zip') as archive:
    names = [entry.filename for entry in archive.infolist()]
    if len(names) != len(allowed) or set(names) != allowed:
        raise RuntimeError('Unexpected or missing reviewed bundle member')
    total = 0
    for entry in archive.infolist():
        name = PurePosixPath(entry.filename)
        if name.is_absolute() or '..' in name.parts or '\\' in entry.filename:
            raise RuntimeError('Unsafe bundle path')
        if entry.is_dir() or entry.file_size > 2_000_000:
            raise RuntimeError('Invalid bundle member')
        total += entry.file_size
        if total > 3_000_000:
            raise RuntimeError('Bundle is too large')
        destination = CODE_ROOT.joinpath(*name.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(entry))
del uploaded
print('VTON worker bundle verified and extracted under /content')


In [ ]:
# Isolate audited Python packages, inherit only the reviewed Colab GPU Torch runtime.
import subprocess, sys, os, json, hashlib
VENV = Path('/content/aiflow-vton-venv')
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(VENV)], check=True)
PYTHON = VENV / 'bin/python'
# Ubuntu Colab may omit ensurepip; inherited pip installs to this venv's sys.prefix.
subprocess.run([str(PYTHON), '-c', "import sys; from pathlib import Path; assert sys.prefix != sys.base_prefix and Path(sys.prefix).resolve() == Path('/content/aiflow-vton-venv').resolve(), 'Wrong installation target'"], check=True)
env = dict(os.environ, PYTHONPATH=str(CODE_ROOT), HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1', HF_HUB_DISABLE_IMPLICIT_TOKEN='1', DIFFUSERS_DISABLE_REMOTE_CODE='1')
subprocess.run([str(PYTHON), '-m', 'colab.vton_worker.bootstrap'], env=env, check=True)
subprocess.run([str(PYTHON), '-c', 'from colab.image_worker.bootstrap import verify_runtime; import json; print(json.dumps(verify_runtime()))'],
               env=env, check=True)


In [ ]:
# Download only three immutable reviewed safetensors/ONNX files (~2.3 GB), then verify every byte.
MODEL_ROOT = Path('/content/aiflow-vton-models')
subprocess.run([str(PYTHON), '-m', 'colab.vton_worker', 'download', '--model-root', str(MODEL_ROOT)], env=env, check=True)


## Start the authenticated API
Supply a fresh private token with at least 32 random ASCII bytes; copy the same token into AIFlow's vton-worker connection form. Use Colab Secrets `AIFLOW_IMAGE_WORKER_TOKEN` or the hidden prompt. Do not store it in Drive or a manifest.

`STORAGE_ROOT` defaults to `/content/aiflow-vton-jobs`. If you choose durable storage, select only a trusted non-code folder: restarting retains completed jobs and marks unfinished jobs failed; it never silently regenerates them.


In [ ]:
if 'worker' in globals() and worker.poll() is None:
    if globals().get('WORKER_KIND') != 'vton':
        raise RuntimeError('Stop the existing SDXL worker or select a fresh runtime before VTON')
    print('Worker already running; keeping the existing process and session token')
else:
    from google.colab import userdata
    from getpass import getpass
    try:
        session_token = userdata.get('AIFLOW_IMAGE_WORKER_TOKEN')
    except Exception:
        session_token = getpass('Private vton-worker Bearer token (32+ random ASCII bytes): ')
    if not session_token or len(session_token.encode()) < 32 or not session_token.isascii() or any(c.isspace() for c in session_token):
        raise RuntimeError('Invalid session token')
    STORAGE_ROOT = Path('/content/aiflow-vton-jobs')
    worker_env = dict(env, AIFLOW_IMAGE_WORKER_TOKEN=session_token, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
    # Model initialization takes several minutes. No access logs, public bind, or token output.
    WORKER_KIND = 'vton'
    log_file = open('/content/aiflow-vton-worker.log', 'w')
    worker = subprocess.Popen([str(PYTHON), '-m', 'colab.vton_worker', 'serve', '--model-root', str(MODEL_ROOT),
                               '--storage', str(STORAGE_ROOT)], env=worker_env, stdout=log_file, stderr=log_file)
    print('Worker starting on localhost:8877; run health check after initialization')


In [ ]:
# Run manually once initialization has finished. Connection refusal means it is still loading or failed.
import urllib.request
request = urllib.request.Request('http://127.0.0.1:8877/v1/health',
                                 headers={'Authorization': 'Bearer ' + session_token})
with urllib.request.urlopen(request, timeout=30) as response:
    health = json.load(response)
if health.get('status') != 'ready' or health.get('api_version') != '1' or 'virtual_try_on' not in health.get('capabilities', []):
    raise RuntimeError('Worker is not ready')
print(json.dumps(health, indent=2))


## Optional real GPU smoke
Run All skips the batch smoke by default. Set `RUN_SMOKE=True` only for a real test; the API worker is stopped first so only one model uses the GPU. Upload one request JSON: `generation_mode="vton"`, `subject_type="human"`, `width=576`, `height=864`, `garment_category="tops"|"bottoms"|"one-pieces"`, `garment_photo_type="flat-lay"`, two PNG/JPEG base64 references with valid SHA256 and ordered roles `person`, `garment`. Guidance defaults to 1.5 and steps to 30. Prompt may be empty because FASHN has no text conditioning. A genuine smoke writes PNG and a receipt with model/source/pose revisions, input/output hashes, precision, hardware, duration and VRAM peaks. Review the generated image; no automatic claim of garment fidelity or identity preservation is made.


In [ ]:
if 'tunnel' in globals() and tunnel.poll() is None:
    print('Tunnel already running; keeping the existing endpoint')
else:
    # Optional audited Cloudflare Quick Tunnel. Every forwarded API route still requires the private Bearer token.
    # Never publish a token in this cell or in its output; enter it only in AIFlow's connection form.
    import urllib.request, hashlib, os
    TUNNEL_BINARY = Path('/content/aiflow-cloudflared')
    TUNNEL_SHA256 = '77e26d8d900e0b8469f416239d14b5f296525fdf79fee6f511ef55609e3fbac2'
    TUNNEL_URL = 'https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64'
    if not TUNNEL_BINARY.exists():
        temporary = TUNNEL_BINARY.with_suffix('.partial')
        size = 0
        with urllib.request.urlopen(TUNNEL_URL, timeout=60) as source, temporary.open('wb') as target:
            while True:
                chunk = source.read(1024 * 1024)
                if not chunk:
                    break
                size += len(chunk)
                if size > 40122749:
                    raise RuntimeError('Tunnel binary exceeds reviewed size')
                target.write(chunk)
        if size != 40122749 or hashlib.sha256(temporary.read_bytes()).hexdigest() != TUNNEL_SHA256:
            raise RuntimeError('Tunnel binary fails reviewed size/hash')
        temporary.replace(TUNNEL_BINARY)
    if hashlib.sha256(TUNNEL_BINARY.read_bytes()).hexdigest() != TUNNEL_SHA256:
        raise RuntimeError('Tunnel binary fails reviewed hash')
    TUNNEL_BINARY.chmod(0o700)
    tunnel_log = open('/content/aiflow-vton-tunnel.log', 'w')
    tunnel = subprocess.Popen([str(TUNNEL_BINARY), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8877'],
                              stdout=tunnel_log, stderr=tunnel_log)
    print('Tunnel starting; run the next cell once to read its endpoint')


In [ ]:
# This log contains only tunnel diagnostics, not request headers or the session token.
import re
endpoints = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/aiflow-vton-tunnel.log').read_text())
if not endpoints:
    raise RuntimeError('Tunnel endpoint not yet available; review tunnel log locally')
print('AIFlow worker URL: ' + endpoints[-1])


In [ ]:
RUN_SMOKE = False  # Change to True only for an explicit batch test; Run All keeps the API running.
if RUN_SMOKE:
    # Optional: run instead of the API when testing an unavailable tunnel.
    # If the API was started above, explicitly stop its own process first.
    if 'worker' in globals() and worker.poll() is None:
        worker.terminate()
        worker.wait(timeout=30)
    smoke_uploaded = files.upload()
    if len(smoke_uploaded) != 1:
        raise RuntimeError('Upload one smoke request JSON')
    smoke_request = Path('/content/aiflow-vton-smoke-request.json')
    smoke_request.write_bytes(next(iter(smoke_uploaded.values())))
    del smoke_uploaded
    subprocess.run([str(PYTHON), '-m', 'colab.vton_worker', 'smoke', '--model-root', str(MODEL_ROOT),
                    '--request', str(smoke_request), '--output', '/content/aiflow-vton-smoke.png'],
                   env=dict(env, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1'), check=True)
    from IPython.display import display, Image as NotebookImage
    display(NotebookImage(filename='/content/aiflow-vton-smoke.png'))
    files.download('/content/aiflow-vton-smoke.png')
    files.download('/content/aiflow-vton-smoke.json')
else:
    print('Optional GPU smoke skipped; worker and tunnel remain running')
